In [ ]:
import numpy as np
import pandas as pd
import statistics

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_validate, train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)


In [ ]:
data = pd.read_csv("../data/data.csv", index_col=0)
print(data)

In [ ]:
numeric_columns = ['lat', 'long', 'solar_angle_elevation', 'month', 'hour', 'temperature_2m',
       'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid',
       'sunshine_duration', 'wind_speed_10m', 'conf',  'laplacian', 'quality', 'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity', 
       'iou', 'lrp', 'pq']
#'precipitation',

categorical_columns = ['time_of_day', 'country', 'road_type', 'road_condition',
       'weather', 'weather_code']

assert len(all_columns) == (len(categorical_columns) + len(numeric_columns)), "Columns not match"


# Assessor Tests

### Baseline

Confidence vs IoU

In [ ]:
baseline = r2_score(data["iou"], data["conf"])
print(f"R2 score of baseline {baseline}")

Random Prediction (Normal dist)

In [ ]:
random_iou = np.random.normal(loc=0.5, scale=0.5, size=len(data))
random_iou = np.clip(random_iou, 0, 1)
random_baseline_r2 = r2_score(data["iou"], random_iou)
print(f"R2 score of random baseline: {random_baseline_r2:.4f}")

### Models

Split data into train 60% and validation 40%

In [ ]:
#X_train, X_test, y_train, y_test = train_test_split(data, test_size=0.4, train_size=0.6, stratify=data["iou"])
X_train, X_test = train_test_split(data, test_size=0.4, train_size=0.6)
y_train = X_train["iou"]
del X_train["iou"]

y_test = X_test["iou"]
del X_test["iou"]

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print(X_train.columns)

In [ ]:
train_numeric_columns = numeric_columns.remove("iou")
train_categorical_columns = categorical_columns

Train models with cv and then test.

#### Linear Regression

In [ ]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), train_numeric_columns),('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), train_categorical_columns)],remainder='passthrough')

linear_reg = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())])
cv_lr = cross_validate(linear_reg, X_train, y_train, cv=10, scoring=("r2", "neg_mean_absolute_error", "neg_mean_squared_error"), return_train_score=True)
print(f"Mean CV train r2 score {statistics.mean(cv_lr["train_r2"])}")
print(f"Mean CV test r2 score {statistics.mean(cv_lr["test_r2"])}")
print(f"Mean CV train MAE score {statistics.mean(cv_lr["train_neg_mean_absolute_error"])}")
print(f"Mean CV test MAE score {statistics.mean(cv_lr["test_neg_mean_absolute_error"])}")
print(f"Mean CV train MSE score {statistics.mean(cv_lr["train_neg_mean_squared_error"])}")
print(f"Mean CV test MSE score {statistics.mean(cv_lr["test_neg_mean_squared_error"])}")

In [ ]:
linear_reg.fit(X_train, y_train)

y_pred = linear_reg.predict(X_test)
lr_test_r2 = r2_score(y_test, y_pred)
print(f"R2 test score {lr_test_r2}")

#### Decision Trees

In [ ]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), train_numeric_columns),('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), train_categorical_columns)],remainder='passthrough')

decision_tree = Pipeline([("preprocessor", preprocessor), ("model", DecisionTreeRegressor())])
cv_dt = cross_validate(decision_tree, X_train, y_train, cv=10, scoring=("r2", "neg_mean_absolute_error", "neg_mean_squared_error"), return_train_score=True)
print(f"Mean CV train r2 score {statistics.mean(cv_dt["train_r2"])}")
print(f"Mean CV test r2 score {statistics.mean(cv_dt["test_r2"])}")
print(f"Mean CV train MAE score {statistics.mean(cv_dt["train_neg_mean_absolute_error"])}")
print(f"Mean CV test MAE score {statistics.mean(cv_dt["test_neg_mean_absolute_error"])}")
print(f"Mean CV train MSE score {statistics.mean(cv_dt["train_neg_mean_squared_error"])}")
print(f"Mean CV test MSE score {statistics.mean(cv_dt["test_neg_mean_squared_error"])}")

In [ ]:
decision_tree.fit(X_train, y_train)
y_pred = decision_tree.predict(X_test)
lr_test_r2 = r2_score(y_test, y_pred)
print(f"R2 test score {lr_test_r2}")

#### Random Forest

In [ ]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), train_numeric_columns),('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), train_categorical_columns)],remainder='passthrough')

rand_forest = Pipeline([("preprocessor", preprocessor), ("model", RandomForestRegressor())])
cv_rf = cross_validate(rand_forest, X_train, y_train, cv=10, scoring=("r2", "neg_mean_absolute_error", "neg_mean_squared_error"), return_train_score=True)
print(f"Mean CV train r2 score {statistics.mean(cv_rf["train_r2"])}")
print(f"Mean CV test r2 score {statistics.mean(cv_rf["test_r2"])}")
print(f"Mean CV train MAE score {statistics.mean(cv_rf["train_neg_mean_absolute_error"])}")
print(f"Mean CV test MAE score {statistics.mean(cv_rf["test_neg_mean_absolute_error"])}")
print(f"Mean CV train MSE score {statistics.mean(cv_rf["train_neg_mean_squared_error"])}")
print(f"Mean CV test MSE score {statistics.mean(cv_rf["test_neg_mean_squared_error"])}")

In [ ]:
rand_forest.fit(X_train, y_train)
y_pred = rand_forest.predict(X_test)
lr_test_r2 = r2_score(y_test, y_pred)
print(f"R2 test score {lr_test_r2}")

#### MLP

In [ ]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), train_numeric_columns),('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), train_categorical_columns)],remainder='passthrough')

mlp = Pipeline([("preprocessor", preprocessor), ("model", MLPRegressor())])
cv_mlp = cross_validate(mlp, X_train, y_train, cv=10, scoring=("r2", "neg_mean_absolute_error", "neg_mean_squared_error"), return_train_score=True)
print(f"Mean CV train r2 score {statistics.mean(cv_mlp["train_r2"])}")
print(f"Mean CV test r2 score {statistics.mean(cv_mlp["test_r2"])}")
print(f"Mean CV train MAE score {statistics.mean(cv_mlp["train_neg_mean_absolute_error"])}")
print(f"Mean CV test MAE score {statistics.mean(cv_mlp["test_neg_mean_absolute_error"])}")
print(f"Mean CV train MSE score {statistics.mean(cv_mlp["train_neg_mean_squared_error"])}")
print(f"Mean CV test MSE score {statistics.mean(cv_mlp["test_neg_mean_squared_error"])}")

In [ ]:
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
lr_test_r2 = r2_score(y_test, y_pred)
print(f"R2 test score {lr_test_r2}")